In [1]:
print("⏳ Đang kiểm tra và cài đặt các gói thư viện cần thiết...")
%pip install -q xgboost scikit-learn torch torch-geometric pandas numpy matplotlib scikit-learn

⏳ Đang kiểm tra và cài đặt các gói thư viện cần thiết...
Note: you may need to restart the kernel to use updated packages.


In [2]:
# === KHỞI TẠO HỆ THỐNG, ĐỒNG BỘ ĐƯỜNG DẪN & NẠP DỮ LIỆU LAI ===
import json
import logging
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import SAGEConv

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score

# 1. Cấu hình hệ thống ghi nhật ký log đồng bộ với các file trước
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

device = torch.device('cpu')
logger.info(f"Hệ thống kích hoạt thiết bị xử lý an toàn: {str(device).upper()}")

# 3. THIẾT LẬP ĐƯỜNG DẪN: Đồng bộ với cấu trúc thư mục cục bộ của cậu
# Nếu notebook nằm chung thư mục gốc, Path(".") sẽ tự động định vị chính xác
PROJECT_DIR = Path(".") 
HYBRID_DIR = PROJECT_DIR / "data" / "gold" / "All_Beauty" / "hybrid_model"
logger.info(f"Đường dẫn kiểm tra gói dữ liệu lai: {HYBRID_DIR.resolve()}")

# 4. Thực thi nạp mảng dữ liệu đặc trưng lai
features = np.load(HYBRID_DIR / "combined_features.npy")
labels = np.load(HYBRID_DIR / "labels.npy")
splits = np.load(HYBRID_DIR / "splits.npy")

# --- CHUẨN HÓA ĐẶC TRƯNG TĨNH ĐỂ CHỐNG SỤP ĐỔ MẠNG NEURAL ---
total_dim = features.shape[1]
text_dim = 384
struct_dim = total_dim - text_dim  # Lấy ra 36 chiều đặc trưng cấu trúc tĩnh thô ban đầu

logger.info(f"Đang chuẩn hóa Gauss cho {struct_dim} chiều đặc trưng tĩnh đầu tiên...")
scaler = StandardScaler()
features[:, :struct_dim] = scaler.fit_transform(features[:, :struct_dim])
logger.info("✓ Hoàn tất chuẩn hóa không gian biên độ đặc trưng nút!")

# Phân tách tập chỉ số theo phân đoạn train/val/test
train_idx = np.where(splits == 0)[0]
val_idx = np.where(splits == 1)[0]
test_idx = np.where(splits == 2)[0]

logger.info(f"✓ Ma trận đặc trưng nút (Node Features): {features.shape}")
logger.info(f"✓ Tập Huấn luyện (Train Set): {len(train_idx):,} mẫu ({len(train_idx)/len(labels)*100:.1f}%)")
logger.info(f"✓ Tập Xác thực (Validation Set): {len(val_idx):,} mẫu ({len(val_idx)/len(labels)*100:.1f}%)")
logger.info(f"✓ Tập Kiểm thử (Test Set): {len(test_idx):,} mẫu ({len(test_idx)/len(labels)*100:.1f}%)")

2026-06-10 00:22:03,730 - INFO - Hệ thống kích hoạt thiết bị xử lý an toàn: CPU
2026-06-10 00:22:03,730 - INFO - Đường dẫn kiểm tra gói dữ liệu lai: E:\MXH\data\gold\All_Beauty\hybrid_model
2026-06-10 00:22:05,086 - INFO - Đang chuẩn hóa Gauss cho 28 chiều đặc trưng tĩnh đầu tiên...
2026-06-10 00:22:05,693 - INFO - ✓ Hoàn tất chuẩn hóa không gian biên độ đặc trưng nút!
2026-06-10 00:22:05,696 - INFO - ✓ Ma trận đặc trưng nút (Node Features): (693929, 412)
2026-06-10 00:22:05,696 - INFO - ✓ Tập Huấn luyện (Train Set): 485,750 mẫu (70.0%)
2026-06-10 00:22:05,699 - INFO - ✓ Tập Xác thực (Validation Set): 104,089 mẫu (15.0%)
2026-06-10 00:22:05,700 - INFO - ✓ Tập Kiểm thử (Test Set): 104,090 mẫu (15.0%)


Baseline Machine Learning

In [3]:
# === CELL 2: HUẤN LUYỆN MÔ HÌNH MỐC BASELINE XGBOOST ===
from xgboost import XGBClassifier

X_train, y_train = features[train_idx], labels[train_idx]
X_test, y_test = features[test_idx], labels[test_idx]

# Tính trọng số phạt phạt mất cân bằng lớp (Lớp 1 chiếm ~76% trong tập kiểm thử của cậu)
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
logger.info(f"Trọng số cân bằng nhãn: 1v{pos_weight:.2f}")

logger.info("🚀 Đang huấn luyện mô hình mốc XGBoost Classifier...")
xgb_baseline = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=pos_weight,
    random_state=42,
    tree_method='hist',
    n_jobs=-1
)
xgb_baseline.fit(X_train, y_train)
logger.info("✓ Hoàn tất huấn luyện mô hình mốc!")

xgb_preds = xgb_baseline.predict(X_test)
xgb_probs = xgb_baseline.predict_proba(X_test)[:, 1]

xgb_f1 = f1_score(y_test, xgb_preds, average='macro') # Dùng macro để cân bằng cả 2 lớp dữ liệu
xgb_auc = roc_auc_score(y_test, xgb_probs)

print("\n📊 [KẾT QUẢ KIỂM THỬ MÔ HÌNH MỐC BASELINE - XGBOOST]:")
print(classification_report(y_test, xgb_preds, digits=4))

2026-06-10 00:22:06,262 - INFO - Trọng số cân bằng nhãn: 1v0.33
2026-06-10 00:22:06,265 - INFO - 🚀 Đang huấn luyện mô hình mốc XGBoost Classifier...
2026-06-10 00:22:16,847 - INFO - ✓ Hoàn tất huấn luyện mô hình mốc!



📊 [KẾT QUẢ KIỂM THỬ MÔ HÌNH MỐC BASELINE - XGBOOST]:
              precision    recall  f1-score   support

           0     0.9716    0.9970    0.9842     24866
           1     0.9991    0.9909    0.9949     79224

    accuracy                         0.9923    104090
   macro avg     0.9853    0.9939    0.9896    104090
weighted avg     0.9925    0.9923    0.9924    104090



PyTorch Geometric Data & Loader

In [4]:
# === CELL 3: DỰNG ĐỒ THỊ PYG TOÀN PHẦN TRÊN BỘ NHỚ RAM ===
from torch_geometric.data import Data

logger.info("Đang trích xuất liên kết cấu trúc đồ thị hành vi từ JSON...")
with open(HYBRID_DIR / "graph_structure.json", "r") as f:
    graph_json = json.load(f)

# Lấy danh sách cạnh đồng hành vi review-review (Cùng sản phẩm + cùng rating + cùng ngày/tháng)
review_edges = graph_json['edges_review_review']['edges']
edge_index = torch.tensor(review_edges, dtype=torch.long).t().contiguous()

# Đóng gói dữ liệu đồ thị lai
x_tensor = torch.tensor(features, dtype=torch.float)
y_tensor = torch.tensor(labels, dtype=torch.long)
graph_data = Data(x=x_tensor, edge_index=edge_index, y=y_tensor)

# Thiết lập mặt nạ Masks phân tách không gian nút
graph_data.train_mask = torch.tensor(splits == 0, dtype=torch.bool)
graph_data.val_mask = torch.tensor(splits == 1, dtype=torch.bool)
graph_data.test_mask = torch.tensor(splits == 2, dtype=torch.bool)

# Chuyển dịch toàn bộ cấu trúc sang thiết bị an toàn (CPU)
graph_data = graph_data.to(device)

logger.info(f"✓ Khởi tạo đồ thị PyG thành công: {graph_data.num_nodes:,} Nút | {graph_data.num_edges:,} Cạnh hành vi")

2026-06-10 00:22:17,065 - INFO - Đang trích xuất liên kết cấu trúc đồ thị hành vi từ JSON...
2026-06-10 00:22:21,422 - INFO - ✓ Khởi tạo đồ thị PyG thành công: 693,929 Nút | 296,494 Cạnh hành vi


GraphSAGE Model Architecture

In [5]:
# === CELL 4: KIẾN TRÚC GRAPHSAGE LAI TỐI ƯU HÓA BỘ NHỚ CHỐNG OVER-SMOOTHING ===
from torch_geometric.nn import SAGEConv

class GraphSAGEWithSkipEfficient(torch.nn.Module):
    """Mạng GraphSAGE tích hợp đường kết nối tắt trực tiếp, triệt tiêu lệnh sao chép bộ nhớ"""
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        
        # Tầng phân loại tuyến tính nhận đầu vào kết hợp giữa cấu trúc đồ thị (hidden) và bản sắc nút gốc (in_channels)
        self.classifier = nn.Linear(hidden_channels + in_channels, out_channels)
        
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        # 1. Lan truyền thông tin qua tầng tích chập đồ thị 1
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=0.3, training=self.training)
        
        # 2. Lan truyền thông tin qua tầng tích chập đồ thị 2
        h = self.conv2(h, edge_index)
        h = F.relu(h)
        
        # 3. KẾT NỐI TẮT HIỆU QUẢ: Nối trực tiếp đặc trưng láng giềng h và x gốc bằng tham chiếu tensor
        return self.classifier(torch.cat([h, x], dim=1))

# Khởi tạo mô hình trên CPU
gnn_model = GraphSAGEWithSkipEfficient(
    in_channels=graph_data.num_features, 
    hidden_channels=128, 
    out_channels=2
).to(device)

print("📐 CẤU TRÚC MẠNG GRAPHSAGE LAI KHÔI PHỤC TÍN HIỆU:")
print(gnn_model)

📐 CẤU TRÚC MẠNG GRAPHSAGE LAI KHÔI PHỤC TÍN HIỆU:
GraphSAGEWithSkipEfficient(
  (conv1): SAGEConv(412, 128, aggr=mean)
  (conv2): SAGEConv(128, 128, aggr=mean)
  (classifier): Linear(in_features=540, out_features=2, bias=True)
)


Model Training & Validation Loop

In [6]:
# === CELL 5: CHU TRÌNH HUẤN LUYỆN GRAPHSAGE TRÊN BỘ NHỚ AN TOÀN ===
class_weights = torch.tensor([1.0, float(pos_weight)], dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(gnn_model.parameters(), lr=0.001, weight_decay=5e-4)

NUM_EPOCHS = 100  

logger.info("🔥 Bắt đầu tiến trình huấn luyện GraphSAGE Lai (Full-Batch CPU)...")
for epoch in range(1, NUM_EPOCHS + 1):
    gnn_model.train()
    optimizer.zero_grad()
    
    # Forward pass tính toán biểu diễn không gian đồ thị
    out = gnn_model(graph_data.x, graph_data.edge_index)
    loss = criterion(out[graph_data.train_mask], graph_data.y[graph_data.train_mask])
    
    # Backward pass chạy ổn định trên không gian RAM hệ thống
    loss.backward()
    optimizer.step()
    
    # Theo dõi tiến trình hội tụ định kỳ sau mỗi 10 Epoch
    if epoch % 10 == 0 or epoch == 1:
        gnn_model.eval()
        with torch.no_grad():
            val_out = gnn_model(graph_data.x, graph_data.edge_index)
            val_loss = criterion(val_out[graph_data.val_mask], graph_data.y[graph_data.val_mask])
        logger.info(f"Epoch {epoch:03d}/{NUM_EPOCHS} -> Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

logger.info("✅ Hoàn tất chu trình huấn luyện mô hình GNN đề xuất!")

2026-06-10 00:22:21,677 - INFO - 🔥 Bắt đầu tiến trình huấn luyện GraphSAGE Lai (Full-Batch CPU)...
2026-06-10 00:22:51,174 - INFO - Epoch 001/100 -> Train Loss: 0.6997 | Val Loss: 0.6956
2026-06-10 00:25:12,452 - INFO - Epoch 010/100 -> Train Loss: 0.6514 | Val Loss: 0.6441
2026-06-10 00:28:06,732 - INFO - Epoch 020/100 -> Train Loss: 0.5782 | Val Loss: 0.5873
2026-06-10 00:30:46,748 - INFO - Epoch 030/100 -> Train Loss: 0.4901 | Val Loss: 0.5056
2026-06-10 00:33:25,489 - INFO - Epoch 040/100 -> Train Loss: 0.3759 | Val Loss: 0.3963
2026-06-10 00:36:12,704 - INFO - Epoch 050/100 -> Train Loss: 0.2629 | Val Loss: 0.2762
2026-06-10 00:38:58,194 - INFO - Epoch 060/100 -> Train Loss: 0.1719 | Val Loss: 0.1750
2026-06-10 00:41:46,899 - INFO - Epoch 070/100 -> Train Loss: 0.1110 | Val Loss: 0.1061
2026-06-10 00:44:38,290 - INFO - Epoch 080/100 -> Train Loss: 0.0755 | Val Loss: 0.0674
2026-06-10 00:47:18,574 - INFO - Epoch 090/100 -> Train Loss: 0.0568 | Val Loss: 0.0484
2026-06-10 00:50:02,1

Kiểm thử và Xuất bảng đối sánh thực nghiệm

In [7]:
# === CELL 6: KIỂM THỬ ĐỘC LẬP VÀ XUẤT BẢNG ĐỐI SÁNH HIỆU NĂNG ===
gnn_model.eval()

logger.info("Đang thực hiện suy luận dự đoán trên tập kiểm thử độc lập...")
with torch.no_grad():
    out = gnn_model(graph_data.x, graph_data.edge_index)
    test_logits = out[graph_data.test_mask]
    
    gnn_probs = F.softmax(test_logits, dim=1)[:, 1].numpy()
    gnn_preds = test_logits.argmax(dim=1).numpy()
    true_labels = graph_data.y[graph_data.test_mask].numpy()

gnn_f1 = f1_score(true_labels, gnn_preds, average='macro')
gnn_auc = roc_auc_score(true_labels, gnn_probs)

print("\n📊 [KẾT QUẢ KIỂM THỬ MÔ HÌNH ĐỀ XUẤT GRAPHSAGE LAI + SEMANTICS]:")
print(classification_report(true_labels, gnn_preds, digits=4))

# === XÂY DỰNG BẢNG ĐỐI SÁNH HIỆU NĂNG HỆ THỐNG PHÂN LOẠI ===
comparison_df = pd.DataFrame({
    'Mô hình thực nghiệm': ['Baseline (XGBoost Classifier)', 'Proposed Model (GraphSAGE Lai)'],
    'F1-Score (Macro Average)': [f"{xgb_f1*100:.2f}%", f"{gnn_f1*100:.2f}%"],
    'AUC-ROC (Độ phân tách)': [f"{xgb_auc:.4f}", f"{gnn_auc:.4f}"]
})

print("\n📈 " + "="*20 + " BẢNG ĐỐI SÁNH HIỆU NĂNG HỆ THỐNG GIAI ĐOẠN 3 " + "="*20)
print(comparison_df.to_string(index=False))
print("="*85)

2026-06-10 00:50:02,512 - INFO - Đang thực hiện suy luận dự đoán trên tập kiểm thử độc lập...



📊 [KẾT QUẢ KIỂM THỬ MÔ HÌNH ĐỀ XUẤT GRAPHSAGE LAI + SEMANTICS]:
              precision    recall  f1-score   support

           0     0.9671    0.9971    0.9818     24866
           1     0.9991    0.9893    0.9942     79224

    accuracy                         0.9912    104090
   macro avg     0.9831    0.9932    0.9880    104090
weighted avg     0.9914    0.9912    0.9912    104090


📈 ==================== BẢNG ĐỐI SÁNH HIỆU NĂNG HỆ THỐNG GIAI ĐOẠN 3 ====================
           Mô hình thực nghiệm F1-Score (Macro Average) AUC-ROC (Độ phân tách)
 Baseline (XGBoost Classifier)                   98.96%                 0.9993
Proposed Model (GraphSAGE Lai)                   98.80%                 0.9980
